## WTF RNN

- Stage 1: Download WikiText-2 and inspect raw text.
- Stage 2: Tokenize it ourselves and understand the vocabulary.
- Stage 3: Build stoi / itos.
- Stage 4: Convert real text → token IDs.
- Stage 5: Create input/target sequences.
- Stage 6: Dataset + DataLoader.
- Stage 7: Build our RNN class ourselves using Wxh, Whh, tanh.
- Stage 8: Train it and watch loss decrease.
- Stage 9: Give it something like "the history of" and let it generate text one token at a time.

### stage-1

In [34]:
from datasets import load_dataset


dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)

print(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [35]:
train_data = dataset["train"]

In [36]:
print(train_data[:10])

{'text': ['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game m

In [37]:
for i in range(20):
    print(i, repr(train_data[i]["text"]))

0 ''
1 ' = Valkyria Chronicles III = \n'
2 ''
3 ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
4 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more 

### stage-2

In [38]:
import re


def tokenize(text):
    text = text.lower()

    tokens = re.findall(
        r"\w+|[^\w\s]",
        text
    )

    return tokens

In [39]:
text = "The dog was running through the field."

tokens = tokenize(text)

print(tokens)

['the', 'dog', 'was', 'running', 'through', 'the', 'field', '.']


In [40]:
text = "AI is amazing! Isn't it?"

print(tokenize(text))

['ai', 'is', 'amazing', '!', 'isn', "'", 't', 'it', '?']


In [41]:
all_tokens = []

for row in train_data:
    text = row["text"]
    tokens = tokenize(text)
    all_tokens.extend(tokens)

In [42]:
print(all_tokens[:100])

['=', 'valkyria', 'chronicles', 'iii', '=', 'senjō', 'no', 'valkyria', '3', ':', 'unrecorded', 'chronicles', '(', 'japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'valkyria', 'of', 'the', 'battlefield', '3', ')', ',', 'commonly', 'referred', 'to', 'as', 'valkyria', 'chronicles', 'iii', 'outside', 'japan', ',', 'is', 'a', 'tactical', 'role', '@', '-', '@', 'playing', 'video', 'game', 'developed', 'by', 'sega', 'and', 'media', '.', 'vision', 'for', 'the', 'playstation', 'portable', '.', 'released', 'in', 'january', '2011', 'in', 'japan', ',', 'it', 'is', 'the', 'third', 'game', 'in', 'the', 'valkyria', 'series', '.', 'employing', 'the', 'same', 'fusion', 'of', 'tactical', 'and', 'real', '@', '-', '@', 'time', 'gameplay', 'as', 'its', 'predecessors', ',', 'the', 'story', 'runs', 'parallel', 'to', 'the', 'first', 'game']


In [43]:
print("Number of tokens:", len(all_tokens))

Number of tokens: 2122137


In [44]:
unique_tokens = set(all_tokens)

print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(unique_tokens))

Total tokens: 2122137
Unique tokens: 65989


In [45]:
from collections import Counter

token_counts = Counter(all_tokens)

print(token_counts.most_common(20))

[('the', 130771), (',', 102624), ('.', 84291), ('of', 57032), ('and', 50738), ('@', 45600), ('in', 45019), ('to', 39522), ('a', 36567), ('=', 29570), ('"', 28309), ('was', 21008), ("'", 18655), ('-', 17337), ('on', 15141), ('as', 15058), ('s', 14982), ('that', 14351), ('for', 13795), ('with', 13012)]


In [46]:
max_vocab_size = 5000

In [47]:
special_tokens = [
    "<PAD>",
    "<UNK>"
]

In [48]:
max_vocab_size = 5000

most_common_tokens = token_counts.most_common(
    max_vocab_size - 2
)

In [49]:
vocab_tokens = [
    token
    for token, count in most_common_tokens
]

In [50]:
vocab_tokens

['the',
 ',',
 '.',
 'of',
 'and',
 '@',
 'in',
 'to',
 'a',
 '=',
 '"',
 'was',
 "'",
 '-',
 'on',
 'as',
 's',
 'that',
 'for',
 'with',
 'by',
 ')',
 '(',
 'is',
 'it',
 'from',
 'at',
 'his',
 'he',
 'were',
 'an',
 'had',
 'which',
 'be',
 'are',
 'this',
 'their',
 'first',
 'but',
 ';',
 'not',
 '–',
 'one',
 'they',
 'its',
 'also',
 ':',
 'after',
 'her',
 'or',
 'two',
 'have',
 'has',
 'been',
 'who',
 'she',
 'new',
 'other',
 'during',
 'when',
 'time',
 'all',
 'into',
 'more',
 'would',
 '1',
 'i',
 'over',
 'while',
 'game',
 'only',
 'most',
 '2',
 'three',
 'later',
 'about',
 'up',
 'may',
 'between',
 'him',
 'song',
 'there',
 'some',
 'than',
 'out',
 'no',
 'season',
 'year',
 'made',
 '3',
 'city',
 'such',
 'before',
 'where',
 'used',
 'series',
 'them',
 'second',
 'world',
 'being',
 'years',
 'both',
 '000',
 'many',
 'these',
 'film',
 'however',
 'album',
 'south',
 '5',
 'war',
 'through',
 'north',
 'then',
 'part',
 'can',
 'early',
 '4',
 'several',
 

In [51]:
vocab = [
    "<PAD>",
    "<UNK>"
] + vocab_tokens

In [52]:
print(vocab[:20])
print("Vocabulary size:", len(vocab))

['<PAD>', '<UNK>', 'the', ',', '.', 'of', 'and', '@', 'in', 'to', 'a', '=', '"', 'was', "'", '-', 'on', 'as', 's', 'that']
Vocabulary size: 5000


In [53]:
stoi = {
    token: index
    for index, token in enumerate(vocab)
}

In [54]:
print(stoi["the"])
print(stoi["<UNK>"])

2
1


In [55]:
itos = {
    index: token
    for token, index in stoi.items()
}

In [56]:
print(itos[2])

the


In [57]:
sentence = "the computer loves artificial intelligence"

tokens = tokenize(sentence)

print("Tokens:")
print(tokens)

Tokens:
['the', 'computer', 'loves', 'artificial', 'intelligence']


In [58]:
unk_id = stoi["<UNK>"]

token_ids = [
    stoi.get(token, unk_id)
    for token in tokens
]

print("Token IDs:")
print(token_ids)

Token IDs:
[2, 1597, 1, 3196, 2027]


In [59]:
decoded = [
    itos[token_id]
    for token_id in token_ids
]

print("Decoded:")
print(decoded)

Decoded:
['the', 'computer', '<UNK>', 'artificial', 'intelligence']


In [ ]:
import torch
from torch.utils.data import Dataset


class RNNDataset(Dataset):

    def __init__(self, token_ids, sequence_length):
        self.token_ids = token_ids
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.token_ids) - self.sequence_length

    def __getitem__(self, index):

        x = self.token_ids[
            index:index + self.sequence_length
        ]

        y = self.token_ids[
            index + 1:index + self.sequence_length + 1
        ]

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )